# Day 48: Classification: Predicting a Category

**Today's question:** can we predict whether a student will **pass**, and *which kind* of mistakes does the model
make when it is wrong?

Before running a cell marked 🤔, predict what it will show.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split

DATA_DIR = Path("../../../datasets")
df = pd.read_csv(DATA_DIR / "students.csv")
df["has_job_num"] = (df["has_job"] == "yes").astype(int)

## 1. Classes, and one column we must NOT use

In classification the target is a **class** (a category). Here there are two classes: `1` = passed, `0` = failed.

In [ ]:
df["passed"].value_counts()

**Why not use `final_score` as a feature?** Because `passed` is *calculated* from it (`passed = 1` when
`final_score >= 50`). A model given `final_score` would just learn "score ≥ 50" and look perfect. And in real
life, when we know the final score, the exam is over: there is nothing left to predict. Using information that
would not be available at prediction time is called **data leakage** (full lesson on Day 49).

Our features are things we know **before** the exam:

In [ ]:
features = ["study_hours", "attendance_pct", "sleep_hours", "previous_score", "assignments_done", "has_job_num"]
X = df[features]
y = df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("baseline accuracy:", baseline.score(X_test, y_test))

Same 60 test students as all week, so the score to beat is still **0.65**.

## 2. Logistic regression: a line that separates

Despite its name, `LogisticRegression` is a **classification** model. Intuition: it combines the features into
one "pass score" (like yesterday's line), then squeezes that score into a **probability between 0 and 1**.
Above 0.5 → predict pass, below → predict fail.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_predictions = log_model.predict(X_test)
print(log_predictions[:10])

`max_iter=1000` gives the model more attempts to settle on its best line. Without it you may see a
`ConvergenceWarning` on other datasets.

**Accuracy** = fraction of test students classified correctly.

In [ ]:
from sklearn.metrics import accuracy_score

print("baseline accuracy:", baseline.score(X_test, y_test))
print("logistic accuracy:", accuracy_score(y_test, log_predictions))

0.85: right about 51 of 60 students, versus 39 for the baseline.

## 3. How confident? `predict_proba`

`predict_proba` gives, for each student, the probability of each class: column 0 = fail, column 1 = pass.

In [ ]:
probabilities = log_model.predict_proba(X_test)
print(probabilities[:5].round(2))

Easier to read in a table. `[:, 1]` takes column 1 (the pass probability) for every row (Day 43 2-D indexing).

In [ ]:
confidence = pd.DataFrame({
    "actual": y_test,
    "predicted": log_predictions,
    "P(pass)": probabilities[:, 1].round(2),
})
confidence.head()

🤔 Which of these five students did the model get **wrong**? How sure was it?

Student 264 actually passed, but the model gave only a 23% chance of passing. A probability of 0.23 does not mean
"impossible": roughly 1 in 4 students like this still pass. Probabilities near 0.5 mean "the model is unsure";
near 0 or 1 mean "the model is confident" (which is not the same as "right").

## 4. A decision tree: a tree of yes/no questions

On Day 46 the tree was just a threshold finder. Here is the full picture: each box asks a yes/no question;
**left = yes, right = no**; the bottom boxes (leaves) give the answer.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_predictions = tree.predict(X_test)
print("tree accuracy:", accuracy_score(y_test, tree_predictions))

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=features, class_names=["fail", "pass"], impurity=False, fontsize=9)
plt.show()

How to read one box: `samples` = training students who reach this box; `value = [fails, passes]` among them;
`class` = the majority, which is the prediction.

🤔 Follow the tree for **Fatima**: previous score 60, studies 9 hours, attendance 85%, no job. Which leaf?

In [ ]:
fatima = pd.DataFrame([{
    "study_hours": 9, "attendance_pct": 85, "sleep_hours": 7,
    "previous_score": 60, "assignments_done": 8, "has_job_num": 0,
}])
print("tree says:", tree.predict(fatima))
print("P(fail), P(pass):", tree.predict_proba(fatima).round(2))

A tree's probability is simply the share of passers in that leaf among training students.

**Small `max_depth` keeps the tree readable.** With unlimited depth it asks dozens of questions and memorises
the training students (Day 49).

## 5. Accuracy hides *which* mistakes: the confusion matrix

Both models make mistakes. But a model that misses struggling students is not the same as one that worries about
students who are fine. The **confusion matrix** counts each kind.

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, log_predictions))

**Rows = what really happened. Columns = what the model predicted.** (Always `confusion_matrix(y_test, predictions)`:
actual first.)

| | predicted fail (0) | predicted pass (1) |
|-|-----------------|-----------------|
| **actually failed (0)** | **15**: failing students correctly flagged | **6**: failing students the model **missed** |
| **actually passed (1)** | **3**: passing students wrongly flagged as fail (**false alarm**) | **36**: passing students correctly predicted |

Read it aloud: *"21 students really failed. The model caught 15 of them and missed 6. Of the 39 who passed, it
wrongly worried about 3."* Accuracy = the diagonal: (15 + 36) / 60 = 0.85.

A picture of the same numbers. Darker = more students.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, log_predictions, display_labels=["fail", "pass"], cmap="Blues")
plt.title("Logistic regression: 6 failing students missed")
plt.show()

🤔 **Predict:** the tree's accuracy was 0.77. Will it miss *more* or *fewer* failing students than 6?

In [ ]:
print(confusion_matrix(y_test, tree_predictions))

The tree missed **9** of the 21 failing students. If our goal is to offer help to students at risk, that is the
cell that matters most, and accuracy alone would never have told us.

## 6. Comparing models in one table

A list of dictionaries (one per model) becomes a tidy results table.

In [ ]:
results = [
    {"model": "baseline (always pass)", "accuracy": baseline.score(X_test, y_test)},
    {"model": "logistic regression", "accuracy": accuracy_score(y_test, log_predictions)},
    {"model": "decision tree (depth 3)", "accuracy": accuracy_score(y_test, tree_predictions)},
]
pd.DataFrame(results).round(3)

## 7. (Optional) Trees can predict numbers too

`DecisionTreeRegressor` asks the same kind of yes/no questions, but each leaf holds the **average score** of
its training students instead of a class. Same pattern, regression metric:

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

y_score = df["final_score"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_score, test_size=0.2, random_state=42)

tree_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_reg.fit(Xr_train, yr_train)
print("tree MAE:", round(mean_absolute_error(yr_test, tree_reg.predict(Xr_test)), 2), "(yesterday's line: 5.04)")

Here the straight line wins: the real pattern in this data is close to "add a bit for each hour, each percent…",
which is exactly what a line describes. Which model is best is always an experiment, never a rule.

## 8. (Optional) More than two classes: iris flowers

scikit-learn ships a few small datasets that need no download. `load_iris` has 150 flowers of **3 species**.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
flowers = iris.frame
print(iris.target_names)
flowers.head(3)

In [ ]:
flower_features = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    flowers[flower_features], flowers["target"], test_size=0.25, random_state=42
)
flower_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
flower_tree.fit(Xi_train, yi_train)
flower_predictions = flower_tree.predict(Xi_test)
print("accuracy:", round(accuracy_score(yi_test, flower_predictions), 3))

ConfusionMatrixDisplay.from_predictions(yi_test, flower_predictions, display_labels=iris.target_names, cmap="Blues")
plt.show()

Same reading rule, now 3 × 3: one versicolor flower was predicted as virginica. Everything else is on the diagonal.

## 9. Common mistakes

In [ ]:
# Mistake 1: a classifier needs classes, but final_score is a continuous number
y_wrong = df["final_score"]
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X, y_wrong, test_size=0.2, random_state=42)
LogisticRegression(max_iter=1000).fit(Xw_train, yw_train)

**Fix:** use a class column (`passed`) for classification, or a regression model for numbers.

**Mistake 2 (no error):** swapping the arguments of `confusion_matrix`. The numbers move to the wrong cells, so
"missed fails" and "false alarms" swap meaning.

In [ ]:
print("correct (actual, predicted):\n", confusion_matrix(y_test, log_predictions))
print("swapped (predicted, actual):\n", confusion_matrix(log_predictions, y_test))

The 6 and the 3 traded places. Habit: **actual first**, the same order as `accuracy_score(y_test, predictions)`.

**Mistake 3 (no error):** celebrating 85% without the baseline. Always show the baseline row in your results table.

## Recap

1. Classification predicts a class. Baseline = always the most common class (0.65 here).
2. `LogisticRegression(max_iter=1000)` turns the features into a probability of passing; `DecisionTreeClassifier`
   asks a few yes/no questions (`max_depth` keeps it small and readable).
3. Accuracy = fraction correct. `predict_proba` = how confident the model is, not how right.
4. Confusion matrix: rows = actual, columns = predicted. Read every cell in words: caught, missed, false alarm.
5. Never use a feature that is only known after the event (`final_score` → `passed`).

**Next time:** why deeper trees do *worse* on new students (overfitting), precision vs recall, and hunting for leaks.